# Task 4.5 — Experiment on repository sizes

Run from the project root: `src/ifm_konwoledge_enginnering/Package_Manager/Project`

In [20]:
import time
from pathlib import Path
import sys
import pandas as pd


SRC = Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from package_manager.use_cases.install import install
from package_manager.domain.verification import verify
from package_manager.infrastructure.solver.knowledge_base import build_knowledge_base


In [14]:
from package_manager.infrastructure.parsing import load_repository, parse_requests_text


def run_one(N):
    repo_path = BASE / f"repo_{N}.json"
    req_path = BASE / f"requests_{N}.txt"

    t0 = time.perf_counter()
    repo = load_repository(repo_path)
    variables, opt, axioms = build_knowledge_base(repo)
    build_time = time.perf_counter() - t0

    requests = parse_requests_text(req_path.read_text())

    t1 = time.perf_counter()
    try:
        installed = install(repo, requests)
        solve_time = time.perf_counter() - t1
        verified = verify(repo, installed, requests) == []
        return {
            "N": N,
            "versions": sum(len(p.versions) for p in repo.packages.values()),
            "axioms": len(axioms),
            "requests": requests,
            "installed": len(installed),
            "build_time": build_time,
            "solve_time": solve_time,
            "verified": verified,
        }
    except Exception as e:
        solve_time = time.perf_counter() - t1
        return {
            "N": N,
            "versions": sum(len(p.versions) for p in repo.packages.values()),
            "axioms": len(axioms),
            "requests": requests,
            "result": f"no solution: {e}",
            "build_time": build_time,
            "solve_time": solve_time,
        }


## Run Example

In [15]:
BASE = Path('examples')
result = run_one('example')

pd.DataFrame.from_dict(result, orient='index')

,0
N,example
versions,20
axioms,26
requests,"[app==2.0, tool]"
result,no solution: no solution for the given requests
build_time,0.007492
solve_time,0.011558


## Run  Experiment

In [16]:
GROUP = "k12"  # change to your group
BASE = Path("data/repos/repos") / GROUP
Ns = [10, 30, 100, 300, 1000]

results = [run_one(N) for N in Ns]

In [17]:
import pandas as pd
pd.DataFrame(results)


,N,versions,axioms,requests,installed,build_time,solve_time,verified,result
0,10,44,22,"[tomli, zipp, comet-sdk]",4.0,0.005785,0.018874,True,NaN
1,30,193,126,"[platformdirs, pyjwt, lumen-api]",9.0,0.031848,0.096546,True,NaN
2,100,1299,1086,"[async-generator, markupsafe-mosaic, umbra-io]",NaN,0.371756,0.604029,NaN,no solution: no solution for the given requests
3,300,6727,6107,"[pyjwt, frozenlist-juniper, nova-core]",4.0,2.971768,5.192908,True,NaN
4,1000,20985,15853,"[annotated-types, h2-alder, sierra-client]",8.0,7.511163,15.528039,True,NaN


# Explain not installed request

In [18]:
from package_manager.use_cases.explain import explain
from package_manager.infrastructure.parsing import parse_repository_text, parse_requests_text



REPO_PATH_TO_EXPLAIN = Path("data/repos/repos") / GROUP / Path("repo_100.json")
REQUEST_PATH_TO_EXPLAIN = Path("data/repos/repos") / GROUP / Path("requests_100.txt")

with open(REPO_PATH_TO_EXPLAIN) as f:
    text = f.read()
    repo = parse_repository_text(text)

with open(REQUEST_PATH_TO_EXPLAIN) as f:
    text = f.read()
    request = parse_requests_text(text)

result = explain(
    repo= repo,
    request_strings=request,
)

for r in result:
    print(r)

at most one package of glyph-cli
Package umbra-io==1.0 requires comet-compat
Package umbra-io==2.0 requires pebble-api
Package umbra-io==2.0 requires ridge-core
Package pebble-api==1.0 requires glyph-cli ==1.0
Package pebble-api==1.1 requires glyph-cli <2
Package ridge-core==1.0 requires glyph-cli >=2.0
comet-compat==1.0 conflicts with umbra-io <2.0
Request #3: umbra-io


In [19]:
print("Request:")
for idx, r in enumerate(request):
    print(f"#{idx+1}: {r}")


print("\nExplain:")
for r in result:
    print(r)

Request:
#1: async-generator
#2: markupsafe-mosaic
#3: umbra-io

Explain:
at most one package of glyph-cli
Package umbra-io==1.0 requires comet-compat
Package umbra-io==2.0 requires pebble-api
Package umbra-io==2.0 requires ridge-core
Package pebble-api==1.0 requires glyph-cli ==1.0
Package pebble-api==1.1 requires glyph-cli <2
Package ridge-core==1.0 requires glyph-cli >=2.0
comet-compat==1.0 conflicts with umbra-io <2.0
Request #3: umbra-io
